In [1]:
import sys
sys.path.append("..")

# 1. Método de doble logística

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from scipy.optimize import curve_fit
from pipeline.ingesta import cargar_indices_desde_bd
from utils.aplicar_whittaker import aplicar_whittaker_series

from pipeline.modulo_fenologico import segmentar_ciclos, detectar_sos
# Ajusta esta ruta al módulo real donde viven las funciones que pegaste
from pipeline.modulo_predictivo import (
    ajustar_curva_doble_logistica,
    extender_serie_con_curva_parametrica,
)

FECHA_INICIO = "2021-01-01"
FECHA_FIN = "2025-12-31"
DISTANCIA_MIN_DIAS = 90
PROMINENCIA_MIN = 0.15
FACTOR_SOS = 0.2
EOS_DIAS = 120  # SOS + 120, tu convención ya establecida

2026-07-07 23:41:13.161 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.163 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.164 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.165 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.166 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.167 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.168 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.169 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-

In [3]:
dfs_raw = cargar_indices_desde_bd(FECHA_INICIO, FECHA_FIN)
dfs_suave = aplicar_whittaker_series(dfs_raw, lambda_param=10000.0, orden=2)

print("Índices disponibles:", list(dfs_raw.keys()))
print("Parcelas:", dfs_raw["EVI"].columns.tolist())

✅  Índices cargados desde BD: 1826 fechas × 9 parcelas (2021-01-01 → 2025-12-31).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...
Índices disponibles: ['EVI', 'LSWI']
Parcelas: ['id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_5', 'id_6', 'id_7', 'id_8']


In [4]:
def obtener_ciclos_con_sos(parcela: str, indice_nombre: str = "EVI") -> list[dict]:
    """
    Replica la lógica de segmentación + detección de SOS de
    graficar_whittaker_sos, pero devuelve los resultados como estructura
    de datos en vez de graficarlos directamente. Cada ciclo detectado
    incluye sus límites (valles) y su fecha SOS si se pudo detectar.
    """
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)

    ciclos = []
    for inicio, fin in segmentos:
        es_ultimo = (inicio, fin) == segmentos[-1]
        mask = (serie.index >= inicio) & (serie.index <= fin) if not es_ultimo else (serie.index >= inicio)
        df_segmento = serie.loc[mask]

        if df_segmento.empty:
            continue

        resultado_sos = detectar_sos(
            serie=df_segmento.values,
            fechas=df_segmento.index,
            factor=FACTOR_SOS,
            metodo="seasonal_amplitude",
            ventana_busqueda=(inicio, serie.index.max()) if es_ultimo else (inicio, fin),
        )
        fecha_sos = resultado_sos.get("sos_fecha")
        if fecha_sos is None:
            continue

        ciclos.append({
            "inicio_valle": inicio,
            "fin_valle": fin,
            "fecha_sos": pd.Timestamp(fecha_sos),
            "fecha_eos": pd.Timestamp(fecha_sos) + pd.Timedelta(days=EOS_DIAS),
        })
    return ciclos

In [5]:
def simular_extrapolacion_ventana(
    parcela: str,
    indice_nombre: str,
    ciclo: dict,
    ventana: str,  # "T1", "T2", "T3"
) -> dict:
    """
    Trunca artificialmente la serie suavizada de un ciclo en el día
    correspondiente a la ventana (30/60/90 post-SOS), extrapola hasta EOS,
    y conserva también la serie real completa del ciclo para poder
    comparar visualmente la extrapolación contra lo que de verdad ocurrió.
    """
    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_completa = dfs_suave[indice_nombre][parcela]
    serie_ciclo_real = serie_completa.loc[fecha_sos:fecha_eos]

    # "Lo que se hubiera conocido" hasta la ventana T simulada
    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    if fecha_corte not in serie_ciclo_real.index:
        return None  # el ciclo aún no llega a esta ventana, no se puede simular

    serie_extendida, params = extender_serie_con_curva_parametrica(
        serie_suavizada=serie_truncada,
        fecha_sos=fecha_sos,
        fecha_fin_extension=fecha_eos,
    )

    # Solo la porción extrapolada (posterior al corte), para graficarla aparte
    serie_solo_extrapolada = serie_extendida.loc[fecha_corte:]

    return {
        "fecha_corte": fecha_corte,
        "fecha_sos": fecha_sos,
        "fecha_eos": fecha_eos,
        "serie_truncada": serie_truncada,
        "serie_extrapolada": serie_solo_extrapolada,
        "serie_real_completa": serie_ciclo_real,
        "params": params,
    }

In [6]:
def graficar_validacion_extrapolacion(parcela: str, indice_nombre: str, ventana: str):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    if not ciclos:
        print("No se detectaron ciclos con SOS válido para esta parcela.")
        return

    opciones_ciclo = {
        f"SOS {c['fecha_sos'].date()} → EOS {c['fecha_eos'].date()}": c
        for c in ciclos
    }

    def actualizar(ciclo_label):
        ciclo = opciones_ciclo[ciclo_label]
        resultado = simular_extrapolacion_ventana(parcela, indice_nombre, ciclo, ventana)

        if resultado is None:
            print(f"Este ciclo no alcanza la ventana {ventana} (termina antes del día correspondiente).")
            return

        fig = go.Figure()

        # A. Serie real completa del ciclo (referencia, gris tenue de fondo)
        fig.add_trace(go.Scatter(
            x=resultado["serie_real_completa"].index,
            y=resultado["serie_real_completa"].values,
            mode="lines",
            line=dict(color="lightgray", width=2),
            name="Serie real completa (ground truth histórico)",
        ))

        # B. Porción "conocida" hasta la ventana T (sólido azul)
        fig.add_trace(go.Scatter(
            x=resultado["serie_truncada"].index,
            y=resultado["serie_truncada"].values,
            mode="lines",
            line=dict(color="#1f77b4", width=2.5),
            name=f"Suavizada observada hasta {ventana}",
        ))

        # C. Extrapolación: discontinua, empieza exactamente en el punto de corte
        serie_extrap = resultado["serie_extrapolada"]
        fig.add_trace(go.Scatter(
            x=serie_extrap.index,
            y=serie_extrap.values,
            mode="lines",
            line=dict(color="#d62728", width=2.5, dash="dash"),
            name="Extrapolación (curva doble logística)",
        ))

        # D. Líneas de referencia: SOS, corte de ventana, EOS
        fig.add_vline(x=ciclo["fecha_sos"], line_width=1.5, line_dash="dot", line_color="green")
        fig.add_annotation(x=ciclo["fecha_sos"], y=1, yref="paper", text="SOS", showarrow=False,
                            textangle=-90, xanchor="right", yanchor="top", font=dict(color="green", size=9))

        fig.add_vline(x=resultado["fecha_corte"], line_width=1.5, line_dash="dot", line_color="#d62728")
        fig.add_annotation(x=resultado["fecha_corte"], y=1, yref="paper", text=f"Corte {ventana}",
                            showarrow=False, textangle=-90, xanchor="right", yanchor="top",
                            font=dict(color="#d62728", size=9))

        fig.add_vline(x=ciclo["fecha_eos"], line_width=1.5, line_dash="dot", line_color="gray")
        fig.add_annotation(x=ciclo["fecha_eos"], y=1, yref="paper", text="EOS (SOS+120)",
                            showarrow=False, textangle=-90, xanchor="right", yanchor="top",
                            font=dict(color="gray", size=9))

        # E. Métrica de calidad del ajuste en el título
        params = resultado["params"]
        r2_txt = f"R²={params['r2']:.3f}" if params is not None else "ajuste no confiable (sin extrapolar)"

        fig.update_layout(
            title=dict(
                text=f"Validación extrapolación: {indice_nombre} en {parcela} — Ventana {ventana} ({r2_txt})",
                font=dict(size=14, weight="bold"),
            ),
            xaxis_title="Fecha",
            yaxis_title=indice_nombre,
            template="plotly_white",
            hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            width=1100,
            height=550,
            margin=dict(t=90, b=40, l=50, r=40),
        )
        fig.show()

    interact(
        actualizar,
        ciclo_label=widgets.Dropdown(options=list(opciones_ciclo.keys()), description="Ciclo:"),
    )

In [ ]:
graficar_validacion_extrapolacion(
    parcela="id_3",       # ajusta al id real de tu parcela
    indice_nombre="EVI",  # o "LSWI"
    ventana="T2",         # "T1", "T2" o "T3"
)

interactive(children=(Dropdown(description='Ciclo:', options=('SOS 2022-08-17 → EOS 2022-12-15', 'SOS 2023-04-…

In [10]:
def calcular_fecha_fin_extension_segura(
    ciclo: dict,
    ciclos_ordenados: list[dict],
    margen_dias: int = 10,
) -> pd.Timestamp:
    """
    Determina la fecha límite de extrapolación para un ciclo, capando
    SOS+120 si el siguiente ciclo detectado arranca antes de esa fecha.
    Evita que la curva doble logística intente modelar el arranque del
    siguiente ciclo, para el cual no tiene forma funcional adecuada.
    """
    idx = ciclos_ordenados.index(ciclo)
    eos_teorico = ciclo["fecha_eos"]  # SOS + 120

    if idx + 1 < len(ciclos_ordenados):
        siguiente_sos = ciclos_ordenados[idx + 1]["fecha_sos"]
        limite_por_contaminacion = siguiente_sos - pd.Timedelta(days=margen_dias)
        return min(eos_teorico, limite_por_contaminacion)

    return eos_teorico

In [11]:
from pipeline.modulo_predictivo import _doble_logistica
def evaluar_plausibilidad_extrapolacion(
    serie_truncada: pd.Series,
    params: dict,
    fecha_sos: pd.Timestamp,
    dias_ventana_tendencia: int = 7,
) -> dict:
    """
    Compara la tendencia local observada (últimos N días antes del corte)
    contra la tendencia que implica la curva ajustada en ese mismo punto.
    Una discrepancia fuerte de signo indica que el modelo unimodal no
    captura la dinámica real (ej. inicio de un nuevo ciclo bleeding-in),
    aunque el R² del ajuste sobre los datos observados sea alto.
    """
    y_recientes = serie_truncada.iloc[-dias_ventana_tendencia:].to_numpy()
    x_recientes = np.arange(len(y_recientes), dtype=float)
    pendiente_observada = np.polyfit(x_recientes, y_recientes, 1)[0]

    ultimo_dia = (serie_truncada.index[-1] - fecha_sos).days
    delta = 1.0
    y0 = _doble_logistica(ultimo_dia, **{k: params[k] for k in ["vmin","vmax","S","mS","A","mA"]})
    y1 = _doble_logistica(ultimo_dia + delta, **{k: params[k] for k in ["vmin","vmax","S","mS","A","mA"]})
    pendiente_modelo = (y1 - y0) / delta

    signos_coinciden = np.sign(pendiente_observada) == np.sign(pendiente_modelo)
    magnitud_razonable = abs(pendiente_modelo) < 3 * abs(pendiente_observada) + 1e-3

    return {
        "pendiente_observada": pendiente_observada,
        "pendiente_modelo": pendiente_modelo,
        "plausible": bool(signos_coinciden and magnitud_razonable),
    }

In [12]:
def validar_extrapolaciones_historicas(
    parcelas: list[str],
    indice_nombre: str,
    ventanas: list[str] = ["T1", "T2", "T3"],
) -> pd.DataFrame:
    """
    Recorre todos los ciclos históricos de las parcelas dadas y calcula
    el error de la extrapolación contra la serie real observada (que en
    modo histórico ya se conoce), para cada ventana T. Permite detectar
    sistemáticamente los casos donde el modelo doble-logístico falla
    (ej. contaminación por ciclo siguiente), en vez de depender de
    inspección visual caso por caso.
    """
    filas = []
    for parcela in parcelas:
        ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
        ciclos_ordenados = sorted(ciclos, key=lambda c: c["fecha_sos"])

        for ciclo in ciclos_ordenados:
            fecha_eos_segura = calcular_fecha_fin_extension_segura(ciclo, ciclos_ordenados)

            for ventana in ventanas:
                resultado = simular_extrapolacion_ventana(parcela, indice_nombre, ciclo, ventana)
                if resultado is None or resultado["params"] is None:
                    continue

                plausibilidad = evaluar_plausibilidad_extrapolacion(
                    resultado["serie_truncada"], resultado["params"], ciclo["fecha_sos"]
                )

                # Comparar extrapolación vs realidad SOLO dentro de la ventana segura
                real_post_corte = resultado["serie_real_completa"].loc[
                    resultado["fecha_corte"]:fecha_eos_segura
                ]
                extrap_post_corte = resultado["serie_extrapolada"].loc[
                    :fecha_eos_segura
                ].reindex(real_post_corte.index)

                error = (extrap_post_corte - real_post_corte).dropna()
                rmse = np.sqrt((error ** 2).mean()) if len(error) > 0 else np.nan

                filas.append({
                    "parcela": parcela,
                    "sos": ciclo["fecha_sos"],
                    "ventana": ventana,
                    "r2_ajuste": resultado["params"]["r2"],
                    "rmse_extrapolacion": rmse,
                    "plausible": plausibilidad["plausible"],
                })

    return pd.DataFrame(filas)

In [13]:
df_validacion = validar_extrapolaciones_historicas(
    parcelas=dfs_suave["EVI"].columns.tolist(),
    indice_nombre="EVI",
)
df_validacion.sort_values("rmse_extrapolacion", ascending=False).head(15)

,parcela,sos,ventana,r2_ajuste,rmse_extrapolacion,plausible
162,id_8,2025-05-08,T2,0.999587,0.677324,False
64,id_2,2022-10-04,T2,0.999858,0.672946,True
85,id_3,2024-05-26,T2,1.000000,0.382630,True
26,id_0,2025-04-07,T3,0.998654,0.333292,False
59,id_1,2025-03-25,T3,0.999590,0.310025,True
118,id_6,2022-06-27,T2,0.999997,0.224478,True
21,id_0,2024-09-26,T1,0.999999,0.208955,True
36,id_1,2022-05-28,T1,0.999999,0.205551,True
12,id_0,2023-03-02,T1,1.000000,0.204612,True
112,id_5,2024-06-20,T2,0.999999,0.203384,True


# 2. Método Whittaker + Climatología

In [71]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from whittaker_eilers import WhittakerSmoother

from pipeline.modulo_fenologico import segmentar_ciclos, detectar_sos
from pipeline.modulo_predictivo import (
    ajustar_curva_doble_logistica,
    extender_serie_con_curva_parametrica,
)

In [72]:
FECHA_INICIO = "2021-01-01"
FECHA_FIN = "2025-12-31"

# Alineados con seed_historico_offline (parámetros ya calibrados en producción)
LAMBDA_WHITTAKER = 4000.0
DISTANCIA_MIN_DIAS = 70
PROMINENCIA_MIN = 0.05
FACTOR_SOS = 0.2
ORDEN_WHITTAKER = 2

# Parámetros del segmento de extrapolación (sin cambios)
LAMBDA_EXTRAPOLACION = 5.0
PESO_INICIAL_ANCLA = 3.0
PESO_PERFIL_FENOLOGICO = 0.5

In [73]:
dfs_raw = cargar_indices_desde_bd(FECHA_INICIO, FECHA_FIN)
dfs_suave = aplicar_whittaker_series(dfs_raw, lambda_param=LAMBDA_WHITTAKER, orden=ORDEN_WHITTAKER)


def obtener_ciclos_con_sos(parcela: str, indice_nombre: str = "EVI") -> list[dict]:
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)
    ciclos = []
    for inicio, fin in segmentos:
        es_ultimo = (inicio, fin) == segmentos[-1]
        mask = (serie.index >= inicio) & (serie.index <= fin) if not es_ultimo else (serie.index >= inicio)
        df_segmento = serie.loc[mask]
        if df_segmento.empty:
            continue
        resultado_sos = detectar_sos(
            serie=df_segmento.values, fechas=df_segmento.index, factor=FACTOR_SOS,
            metodo="seasonal_amplitude",
            ventana_busqueda=(inicio, serie.index.max()) if es_ultimo else (inicio, fin),
        )
        fecha_sos = resultado_sos.get("sos_fecha")
        if fecha_sos is None:
            continue
        fecha_sos = pd.Timestamp(fecha_sos)
        ciclos.append({
            "inicio_valle": inicio, "fin_valle": fin,
            "fecha_sos": fecha_sos, "fecha_eos": fecha_sos + pd.Timedelta(days=EOS_DIAS),
        })
    return sorted(ciclos, key=lambda c: c["fecha_sos"])

✅  Índices cargados desde BD: 1826 fechas × 9 parcelas (2021-01-01 → 2025-12-31).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...


In [77]:
def obtener_ciclos_con_sos(parcela: str, indice_nombre: str = "EVI") -> list[dict]:
    """
    Segmenta la serie suavizada en ciclos y detecta SOS por ciclo, usando
    la misma convención que seed_historico_offline: fecha_eos es el valle
    de fin de segmento detectado por segmentar_ciclos (fin), NO
    fecha_sos + 120 días. Esta es la definición real persistida en
    produccion_acumulada_ciclo vía crear_ciclo_historico(sos_fecha,
    eos_fecha=fin).
    """
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)

    ciclos = []
    for inicio, fin in segmentos:
        es_ultimo = (inicio, fin) == segmentos[-1]
        mask = (serie.index >= inicio) & (serie.index <= fin) if not es_ultimo else (serie.index >= inicio)
        df_segmento = serie.loc[mask]
        if df_segmento.empty:
            continue

        resultado_sos = detectar_sos(
            serie=df_segmento.values,
            fechas=df_segmento.index,
            factor=FACTOR_SOS,
            metodo="seasonal_amplitude",
            ventana_busqueda=(inicio, serie.index.max()) if es_ultimo else (inicio, fin),
        )
        fecha_sos = resultado_sos.get("sos_fecha")
        if fecha_sos is None:
            continue

        fecha_sos = pd.Timestamp(fecha_sos)
        ciclos.append({
            "inicio_valle": inicio,
            "fin_valle": fin,
            "fecha_sos": fecha_sos,
            "fecha_eos": pd.Timestamp(fin),  # ← corregido: EOS real = valle de fin, no SOS+120
        })
    return sorted(ciclos, key=lambda c: c["fecha_sos"])

In [78]:
def diagnosticar_duraciones_ciclos(parcela: str, indice_nombre: str = "EVI"):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    for c in ciclos:
        duracion = (c["fecha_eos"] - c["fecha_sos"]).days
        alerta = "⚠️ menor a 90 días (T3 no aplicable)" if duracion < 90 else ""
        print(f"SOS {c['fecha_sos'].date()} → EOS {c['fecha_eos'].date()}  |  duración: {duracion} días  {alerta}")

diagnosticar_duraciones_ciclos("id_1", "EVI")

SOS 2021-07-22 → EOS 2021-10-06  |  duración: 76 días  ⚠️ menor a 90 días (T3 no aplicable)
SOS 2021-10-18 → EOS 2022-01-12  |  duración: 86 días  ⚠️ menor a 90 días (T3 no aplicable)
SOS 2022-01-28 → EOS 2022-05-06  |  duración: 98 días  
SOS 2022-05-20 → EOS 2022-07-31  |  duración: 72 días  ⚠️ menor a 90 días (T3 no aplicable)
SOS 2022-08-17 → EOS 2022-11-15  |  duración: 90 días  
SOS 2022-11-26 → EOS 2023-01-27  |  duración: 62 días  ⚠️ menor a 90 días (T3 no aplicable)
SOS 2023-02-20 → EOS 2023-05-22  |  duración: 91 días  
SOS 2023-06-10 → EOS 2023-09-29  |  duración: 111 días  
SOS 2023-10-13 → EOS 2024-01-20  |  duración: 99 días  
SOS 2024-02-05 → EOS 2024-05-18  |  duración: 103 días  
SOS 2024-06-01 → EOS 2024-09-01  |  duración: 92 días  
SOS 2024-09-24 → EOS 2025-01-25  |  duración: 123 días  
SOS 2025-03-26 → EOS 2025-08-21  |  duración: 148 días  


In [74]:
def calcular_perfil_fenologico_referencia(
    indice_nombre: str,
    parcelas: list[str] | None = None,
    metodo_agregacion: str = "median",
) -> pd.Series:
    """
    Perfil estacional de referencia por dia_anio (1-366), derivado de las
    series suavizadas históricas de Sentinel-2 (dfs_suave) — NO es la
    climatología meteorológica de climatologia_diaria (PAR/temperatura,
    AgERA5). Es autoreferencial al propio índice espectral: agrega los
    valores observados en años previos para cada día calendario, across
    parcelas, capturando el patrón bimodal primera/postrera si hay
    suficiente profundidad histórica.
    """
    df = dfs_suave[indice_nombre]
    columnas = parcelas if parcelas is not None else df.columns.tolist()
    df_dia_anio = df[columnas].copy()
    df_dia_anio["dia_anio"] = df_dia_anio.index.dayofyear

    if metodo_agregacion == "median":
        perfil = df_dia_anio.groupby("dia_anio")[columnas].median().median(axis=1)
    else:
        perfil = df_dia_anio.groupby("dia_anio")[columnas].mean().mean(axis=1)

    perfil.name = "valor_referencia"
    return perfil


def graficar_perfil_fenologico(perfil: pd.Series, indice_nombre: str):
    """Verificación visual OBLIGATORIA antes de usar el perfil como ancla:
    confirma que muestra el patrón bimodal esperado (primera + postrera)."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=perfil.index, y=perfil.values, mode="lines+markers",
                              line=dict(color="#9467bd", width=2)))
    fig.update_layout(
        title=f"Perfil fenológico de referencia — {indice_nombre} (agregado histórico por día del año)",
        xaxis_title="Día del año", yaxis_title=indice_nombre,
        template="plotly_white", width=1000, height=400,
    )
    fig.show()

In [75]:
perfil_evi = calcular_perfil_fenologico_referencia("EVI")
graficar_perfil_fenologico(perfil_evi, "EVI")

In [76]:
def whittaker_con_pesos_personalizados(
    serie_valores: np.ndarray, pesos: np.ndarray,
    lambda_param: float, orden: int = ORDEN_WHITTAKER,
) -> np.ndarray:
    valores_preparados = np.nan_to_num(serie_valores, nan=0.0).tolist()
    num_validos = int(np.sum(pesos > 0))
    if num_validos < (orden + 1):
        return pd.Series(serie_valores).interpolate(
            method="linear", limit_direction="both"
        ).fillna(0.0).to_numpy()
    whittaker = WhittakerSmoother(
        lmbda=lambda_param, order=orden, data_length=len(valores_preparados),
        x_input=None, weights=pesos.tolist(),
    )
    return np.array(whittaker.smooth(valores_preparados))

In [79]:
def construir_segmento_extrapolado_perfil(
    serie_truncada: pd.Series,
    fecha_fin_extension: pd.Timestamp,
    perfil_referencia: pd.Series,
    peso_inicial: float = PESO_INICIAL_ANCLA,
    peso_perfil: float = PESO_PERFIL_FENOLOGICO,
    espaciado_dias: int = 1,
    lambda_extrapolacion: float = LAMBDA_EXTRAPOLACION,
) -> pd.Series:
    """
    Suaviza SOLO el tramo posterior al corte. El primer punto es el
    último valor observado real, anclado con peso alto para continuidad
    exacta. Las anclas posteriores vienen del perfil fenológico de
    referencia (histórico propio, no meteorológico).
    """
    fechas = pd.date_range(serie_truncada.index[-1], fecha_fin_extension, freq="D")
    valores = np.full(len(fechas), np.nan)
    pesos = np.zeros(len(fechas))
    valores[0] = serie_truncada.iloc[-1]
    pesos[0] = peso_inicial

    for i in range(1, len(fechas)):
        if i % espaciado_dias != 0:
            continue
        dia_anio = fechas[i].dayofyear
        v = perfil_referencia.get(dia_anio, np.nan)
        if not np.isnan(v):
            valores[i] = v
            pesos[i] = peso_perfil

    valores_suaves = whittaker_con_pesos_personalizados(valores, pesos, lambda_param=lambda_extrapolacion)
    return pd.Series(valores_suaves, index=fechas)

In [80]:
def simular_extrapolacion_comparativa(
    parcela: str, indice_nombre: str, ciclo: dict, ventana: str,
    perfil_referencia: pd.Series,
    peso_perfil: float = PESO_PERFIL_FENOLOGICO,
    espaciado_dias: int = 1,
    lambda_extrapolacion: float = LAMBDA_EXTRAPOLACION,
) -> dict | None:
    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_ciclo_real = dfs_suave[indice_nombre][parcela].loc[fecha_sos:fecha_eos]
    if fecha_corte not in serie_ciclo_real.index:
        return None
    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    serie_extendida_dl, params_dl = extender_serie_con_curva_parametrica(
        serie_suavizada=serie_truncada, fecha_sos=fecha_sos, fecha_fin_extension=fecha_eos,
    )
    extrap_dl = serie_extendida_dl.loc[fecha_corte:]

    extrap_perfil = construir_segmento_extrapolado_perfil(
        serie_truncada=serie_truncada, fecha_fin_extension=fecha_eos,
        perfil_referencia=perfil_referencia, peso_perfil=peso_perfil,
        espaciado_dias=espaciado_dias, lambda_extrapolacion=lambda_extrapolacion,
    )

    return {
        "fecha_corte": fecha_corte, "fecha_sos": fecha_sos, "fecha_eos": fecha_eos,
        "serie_truncada": serie_truncada, "serie_real_completa": serie_ciclo_real,
        "extrap_doble_logistica": extrap_dl, "extrap_perfil": extrap_perfil,
        "params_dl": params_dl,
    }

In [81]:
def graficar_comparacion_metodos(parcela: str, indice_nombre: str, ventana: str, perfil_referencia: pd.Series):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    if not ciclos:
        print("No se detectaron ciclos con SOS válido.")
        return

    opciones_ciclo = {f"SOS {c['fecha_sos'].date()} → EOS {c['fecha_eos'].date()}": c for c in ciclos}

    def actualizar(ciclo_label):
        ciclo = opciones_ciclo[ciclo_label]
        r = simular_extrapolacion_comparativa(parcela, indice_nombre, ciclo, ventana, perfil_referencia)
        if r is None:
            print(f"Este ciclo no alcanza la ventana {ventana}.")
            return

        extrap_blend = combinar_extrapolaciones(r["extrap_doble_logistica"], r["extrap_perfil"])

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=r["serie_real_completa"].index, y=r["serie_real_completa"].values,
                                  mode="lines", line=dict(color="lightgray", width=2), name="Real completa"))
        fig.add_trace(go.Scatter(x=r["serie_truncada"].index, y=r["serie_truncada"].values,
                                  mode="lines", line=dict(color="#1f77b4", width=2.5), name=f"Observada hasta {ventana}"))
        fig.add_trace(go.Scatter(x=r["extrap_doble_logistica"].index, y=r["extrap_doble_logistica"].values,
                                  mode="lines", line=dict(color="#d62728", width=2.5, dash="dash"), name="Extrapolación: doble logística"))
        fig.add_trace(go.Scatter(x=r["extrap_perfil"].index, y=r["extrap_perfil"].values,
                                  mode="lines", line=dict(color="#2ca02c", width=2.5, dash="dash"), name="Extrapolación: perfil fenológico + Whittaker"))
        fig.add_trace(go.Scatter(x=extrap_blend.index, y=extrap_blend.values,
                                  mode="lines", line=dict(color="#9467bd", width=2.5, dash="dash"), name="Extrapolación: blend"))

        for fecha, color, texto in [(ciclo["fecha_sos"], "green", "SOS"), (r["fecha_corte"], "#d62728", f"Corte {ventana}"), (ciclo["fecha_eos"], "gray", "EOS")]:
            fig.add_vline(x=fecha, line_width=1.5, line_dash="dot", line_color=color)
            fig.add_annotation(x=fecha, y=1, yref="paper", text=texto, showarrow=False, textangle=-90,
                                xanchor="right", yanchor="top", font=dict(color=color, size=9))

        r2_txt = f"R² doble-log={r['params_dl']['r2']:.3f}" if r["params_dl"] else "doble-log no convergió"
        fig.update_layout(
            title=f"Comparación de extrapolación: {indice_nombre} en {parcela} — {ventana} ({r2_txt})",
            xaxis_title="Fecha", yaxis_title=indice_nombre, template="plotly_white", hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            width=1150, height=550, margin=dict(t=90, b=40, l=50, r=40),
        )
        fig.show()

    interact(actualizar, ciclo_label=widgets.Dropdown(options=list(opciones_ciclo.keys()), description="Ciclo:"))

In [95]:
graficar_comparacion_metodos(parcela="id_5", indice_nombre="EVI", ventana="T1", perfil_referencia=perfil_evi)

interactive(children=(Dropdown(description='Ciclo:', options=('SOS 2021-06-05 → EOS 2021-08-07', 'SOS 2021-08-…

In [83]:
def calcular_error_integrado(
    extrap: pd.Series,
    real: pd.Series,
) -> dict:
    """
    Compara el área bajo la curva extrapolada vs. la real en el tramo
    posterior al corte, usando integración trapezoidal. A diferencia del
    RMSE puntual, esta métrica refleja mejor el efecto acumulado sobre
    GPP/NPP/rendimiento, donde errores de signo opuesto en distintos
    tramos temporales se compensan parcialmente.
    """
    idx_comun = extrap.index.intersection(real.index)
    e = extrap.reindex(idx_comun)
    r = real.reindex(idx_comun)

    dias = (idx_comun - idx_comun[0]).days.to_numpy(dtype=float)

    integrar = getattr(np, "trapezoid", None) or np.trapz

    area_extrap = integrar(e.to_numpy(), dias)
    area_real = integrar(r.to_numpy(), dias)


    bias_integrado = area_extrap - area_real
    error_pct_area = bias_integrado / area_real if area_real != 0 else np.nan

    # Error punto a punto, para comparar contra el integrado
    rmse_puntual = np.sqrt(np.mean((e - r) ** 2))

    return {
        "area_extrapolada": area_extrap,
        "area_real": area_real,
        "bias_integrado": bias_integrado,      # cerca de 0 => cancelación real
        "error_pct_area": error_pct_area,
        "rmse_puntual": rmse_puntual,
    }

In [84]:
def calcular_perfil_fenologico_referencia_por_temporada(
    indice_nombre: str,
    ciclos_por_parcela: dict[str, list[dict]],  # {parcela: [ciclos con fecha_sos, temporada]}
    temporada: str,  # "primera" o "postrera"
    parcelas: list[str] | None = None,
    metodo_agregacion: str = "median",
) -> pd.Series:
    """
    Igual que calcular_perfil_fenologico_referencia, pero filtra las
    observaciones para incluir solo los tramos temporales que
    corresponden a ciclos de una temporada específica, evitando el
    artefacto de mezclar fases fenológicas distintas (primera vs.
    postrera) en el mismo dia_anio agregado.
    """
    df = dfs_suave[indice_nombre]
    columnas = parcelas if parcelas is not None else df.columns.tolist()

    mascaras_temporada = pd.Series(False, index=df.index)
    for parcela, ciclos in ciclos_por_parcela.items():
        if parcela not in columnas:
            continue
        for c in ciclos:
            if c.get("temporada") == temporada:
                mascaras_temporada.loc[c["fecha_sos"]:c["fecha_eos"]] = True

    df_filtrado = df.loc[mascaras_temporada, columnas].copy()
    df_filtrado["dia_anio"] = df_filtrado.index.dayofyear

    if metodo_agregacion == "median":
        perfil = df_filtrado.groupby("dia_anio")[columnas].median().median(axis=1)
    else:
        perfil = df_filtrado.groupby("dia_anio")[columnas].mean().mean(axis=1)

    perfil.name = "valor_referencia"
    return perfil

In [85]:
import sqlite3
from contextlib import closing
from utils.conexionDB import get_connection_raw

def obtener_ciclos_desde_bd(id_parcela: int | None = None) -> pd.DataFrame:
    """
    Trae los ciclos ya persistidos en produccion_acumulada_ciclo, con su
    temporada asignada (primera/postrera), para usarlos como fuente de
    verdad en vez de re-detectar SOS/EOS en memoria en cada notebook.
    """
    query = """
        SELECT id_ciclo, id_parcela, temporada, sos, t1, t2, t3, eos, estado_ciclo
        FROM produccion_acumulada_ciclo
        WHERE sos IS NOT NULL AND eos IS NOT NULL
    """
    params = ()
    if id_parcela is not None:
        query += " AND id_parcela = ?"
        params = (id_parcela,)

    with closing(get_connection_raw()) as conn:
        df = pd.read_sql(query, conn, params=params, parse_dates=["sos", "t1", "t2", "t3", "eos"])
    return df

In [86]:
def calcular_perfil_fenologico_por_temporada(
    indice_nombre: str,
    temporada: str,  # 'primera' o 'postrera'
    parcelas: list[str] | None = None,
    metodo_agregacion: str = "median",
) -> pd.Series:
    """
    Perfil fenológico de referencia, filtrado a los tramos SOS-EOS de
    ciclos con la temporada dada, usando la columna temporada real de
    produccion_acumulada_ciclo (no una heurística por mes de SOS).
    """
    ciclos_df = obtener_ciclos_desde_bd()
    ciclos_temporada = ciclos_df[ciclos_df["temporada"] == temporada]

    df = dfs_suave[indice_nombre]
    columnas = parcelas if parcelas is not None else df.columns.tolist()

    mascara = pd.Series(False, index=df.index)
    for _, fila in ciclos_temporada.iterrows():
        col_parcela = f"id_{fila['id_parcela']}"
        if col_parcela not in columnas:
            continue
        mascara.loc[fila["sos"]:fila["eos"]] = True

    df_filtrado = df.loc[mascara, columnas].copy()
    df_filtrado["dia_anio"] = df_filtrado.index.dayofyear

    if metodo_agregacion == "median":
        perfil = df_filtrado.groupby("dia_anio")[columnas].median().median(axis=1)
    else:
        perfil = df_filtrado.groupby("dia_anio")[columnas].mean().mean(axis=1)

    perfil.name = "valor_referencia"
    return perfil

In [87]:
perfil_evi_primera = calcular_perfil_fenologico_por_temporada("EVI", "primera")
perfil_evi_postrera = calcular_perfil_fenologico_por_temporada("EVI", "postrera")

graficar_perfil_fenologico(perfil_evi_primera, "EVI (primera)")
graficar_perfil_fenologico(perfil_evi_postrera, "EVI (postrera)")

In [88]:
def combinar_extrapolaciones(
    extrap_dl: pd.Series,
    extrap_perfil: pd.Series,
    tau_dias: float = 20.0,
) -> pd.Series:
    """
    Combina doble logística y perfil fenológico con peso exponencial
    decreciente hacia el paramétrico conforme se aleja del corte: cerca
    del corte confía en la dinámica local reciente (doble-log), lejos
    confía en el patrón estacional histórico (perfil).
    """
    idx_comun = extrap_dl.index.intersection(extrap_perfil.index)
    dias_desde_corte = (idx_comun - idx_comun[0]).days.to_numpy(dtype=float)
    peso_dl = np.exp(-dias_desde_corte / tau_dias)

    combinado = peso_dl * extrap_dl.reindex(idx_comun).to_numpy() + \
                (1 - peso_dl) * extrap_perfil.reindex(idx_comun).to_numpy()

    return pd.Series(combinado, index=idx_comun)

In [100]:
def validar_metodos_extrapolacion(
    indice_nombre: str,
    ventanas: list[str] = ["T1", "T2", "T3"],
) -> pd.DataFrame:
    """
    Corre los tres métodos (doble-log, perfil por temporada, blend)
    sobre todos los ciclos históricos persistidos en BD, calculando
    error puntual (RMSE) y error integrado (bias de área) para cada
    combinación de parcela x ciclo x ventana. Permite decidir
    cuantitativamente el método final por ventana/temporada.
    """
    ciclos_df = obtener_ciclos_desde_bd()
    filas = []

    perfiles = {
        "primera": calcular_perfil_fenologico_por_temporada(indice_nombre, "primera"),
        "postrera": calcular_perfil_fenologico_por_temporada(indice_nombre, "postrera"),
    }

    for _, fila in ciclos_df.iterrows():
        col_parcela = f"id_{fila['id_parcela']}"
        if col_parcela not in dfs_suave[indice_nombre].columns:
            continue

        temporada = fila["temporada"]
        perfil = perfiles.get(temporada)
        if perfil is None:
            continue

        ciclo = {"fecha_sos": fila["sos"], "fecha_eos": fila["eos"]}

        for ventana in ventanas:
            r = simular_extrapolacion_comparativa(col_parcela, indice_nombre, ciclo, ventana, perfil)
            if r is None or r["params_dl"] is None:
                continue

            real_post = r["serie_real_completa"].loc[r["fecha_corte"]:]

            extrap_blend = combinar_extrapolaciones(r["extrap_doble_logistica"], r["extrap_perfil"])

            for metodo_nombre, extrap in [
                ("doble_logistica", r["extrap_doble_logistica"]),
                ("perfil_temporada", r["extrap_perfil"]),
                ("blend", extrap_blend),
            ]:
                metricas = calcular_error_integrado(extrap, real_post)
                filas.append({
                    "id_parcela": fila["id_parcela"],
                    "id_ciclo": fila["id_ciclo"],   # ← agregado: identifica el ciclo específico
                    "sos": fila["sos"],              # ← agregado: útil para inspección legible
                    "temporada": temporada,
                    "ventana": ventana,
                    "metodo": metodo_nombre,
                    **metricas,
                })

    return pd.DataFrame(filas)

In [101]:
df_val = validar_metodos_extrapolacion("EVI")

# Resumen: qué método gana por ventana y temporada, en error integrado (lo que de verdad importa para GPP/rendimiento)
resumen = df_val.groupby(["temporada", "ventana", "metodo"]).agg(
    bias_integrado_medio=("bias_integrado", "mean"),
    bias_integrado_abs_medio=("bias_integrado", lambda x: x.abs().mean()),
    rmse_puntual_medio=("rmse_puntual", "mean"),
    n_ciclos=("bias_integrado", "count"),
).reset_index()

resumen.sort_values(["temporada", "ventana", "bias_integrado_abs_medio"])

,temporada,ventana,metodo,bias_integrado_medio,bias_integrado_abs_medio,rmse_puntual_medio,n_ciclos
0,postrera,T1,blend,-2.221763,6.731661,0.092680,48
2,postrera,T1,perfil_temporada,-3.809023,8.285858,0.118447,48
1,postrera,T1,doble_logistica,6.227265,8.995214,0.102220,48
3,postrera,T2,blend,-0.707075,4.651050,0.075269,50
5,postrera,T2,perfil_temporada,-0.992378,5.588955,0.104761,50
4,postrera,T2,doble_logistica,-5.777805,10.954505,0.088665,50
6,postrera,T3,blend,-0.169213,3.231465,0.058135,43
8,postrera,T3,perfil_temporada,0.292946,4.106815,0.085542,43
7,postrera,T3,doble_logistica,-5.135841,8.043492,0.082599,43
9,primera,T1,blend,-1.477050,9.463797,0.100755,41


In [102]:
def calcular_perfil_fenologico_ponderado_recencia(
    indice_nombre: str, temporada: str, anio_referencia: int,
    semivida_anios: float = 2.0, parcelas: list[str] | None = None,
) -> pd.Series:
    """Igual que calcular_perfil_fenologico_por_temporada, pero pondera
    cada año con peso exponencial decreciente según su distancia a
    anio_referencia, dando más autoridad a años recientes."""
    ciclos_df = obtener_ciclos_desde_bd()
    ciclos_temporada = ciclos_df[ciclos_df["temporada"] == temporada]
    df = dfs_suave[indice_nombre]
    columnas = parcelas if parcelas is not None else df.columns.tolist()

    registros = []
    for _, fila in ciclos_temporada.iterrows():
        col_parcela = f"id_{fila['id_parcela']}"
        if col_parcela not in columnas:
            continue
        tramo = df.loc[fila["sos"]:fila["eos"], col_parcela]
        anio_ciclo = fila["sos"].year
        peso_anio = 0.5 ** (abs(anio_referencia - anio_ciclo) / semivida_anios)
        for fecha, valor in tramo.items():
            if not np.isnan(valor):
                registros.append({"dia_anio": fecha.dayofyear, "valor": valor, "peso": peso_anio})

    df_reg = pd.DataFrame(registros)
    perfil = df_reg.groupby("dia_anio").apply(
        lambda g: np.average(g["valor"], weights=g["peso"])
    )
    perfil.name = "valor_referencia"
    return perfil

In [103]:
for tau in [10, 20, 35]:
    df_tau = validar_metodos_extrapolacion("EVI")  # necesitarías parametrizar tau dentro de la función
    # comparar bias_integrado_abs_medio del blend por tau

In [104]:
df_tau

,id_parcela,id_ciclo,sos,temporada,ventana,metodo,area_extrapolada,area_real,bias_integrado,error_pct_area,rmse_puntual
0,0,406,2021-01-18,postrera,T1,doble_logistica,67.693743,59.418727,8.275016,0.139266,0.096142
1,0,406,2021-01-18,postrera,T1,perfil_temporada,38.012514,59.418727,-21.406213,-0.360260,0.210967
2,0,406,2021-01-18,postrera,T1,blend,43.155730,59.418727,-16.262998,-0.273702,0.166621
3,0,406,2021-01-18,postrera,T2,doble_logistica,48.451009,43.724758,4.726251,0.108091,0.075033
4,0,406,2021-01-18,postrera,T2,perfil_temporada,30.817310,43.724758,-12.907447,-0.295198,0.182047
...,...,...,...,...,...,...,...,...,...,...,...
754,8,515,2025-09-08,postrera,T2,perfil_temporada,23.841899,23.847814,-0.005914,-0.000248,0.083663
755,8,515,2025-09-08,postrera,T2,blend,24.971189,23.847814,1.123375,0.047106,0.059616
756,8,515,2025-09-08,postrera,T3,doble_logistica,8.550672,7.700576,0.850096,0.110394,0.043367
757,8,515,2025-09-08,postrera,T3,perfil_temporada,9.613746,7.700576,1.913170,0.248445,0.086619


In [110]:
def comparar_metodos_pareado(
    df_val: pd.DataFrame,
    metrica: str = "bias_integrado",
    metodo_a: str = "doble_logistica",
    metodo_b: str = "blend",
) -> pd.DataFrame:
    """
    Comparación pareada por ciclo real (id_ciclo), no por parcela —
    corrige el colapso silencioso que ocurría cuando una parcela tenía
    múltiples ciclos de la misma temporada (2020-2025 = varios años),
    que quedaban reducidos a uno solo bajo aggfunc='first'.
    """
    filas = []
    for (temporada, ventana), grupo in df_val.groupby(["temporada", "ventana"]):
        pivot = grupo.pivot_table(
            index=["id_ciclo"], columns="metodo", values=metrica, aggfunc="first"
        )

        if metodo_a not in pivot.columns or metodo_b not in pivot.columns:
            continue

        pareado = pivot[[metodo_a, metodo_b]].dropna()
        if len(pareado) < 5:
            continue

        err_a = pareado[metodo_a].abs()
        err_b = pareado[metodo_b].abs()

        gana_a = (err_a < err_b).sum()
        gana_b = (err_b < err_a).sum()
        empates = (err_a == err_b).sum()

        try:
            stat, p_valor = stats.wilcoxon(err_a, err_b)
        except ValueError:
            p_valor = np.nan

        filas.append({
            "temporada": temporada,
            "ventana": ventana,
            "n_ciclos_comparados": len(pareado),
            f"gana_{metodo_a}": gana_a,
            f"gana_{metodo_b}": gana_b,
            "empates": empates,
            f"tasa_victoria_{metodo_b}": gana_b / len(pareado),
            f"error_abs_medio_{metodo_a}": err_a.mean(),
            f"error_abs_medio_{metodo_b}": err_b.mean(),
            "diferencia_medias": err_a.mean() - err_b.mean(),
            "p_valor_wilcoxon": p_valor,
            "significativo_al_5pct": p_valor < 0.05 if not np.isnan(p_valor) else None,
        })

    return pd.DataFrame(filas)

In [111]:
comparacion_bias = comparar_metodos_pareado(df_val, metrica="bias_integrado", metodo_a="doble_logistica", metodo_b="blend")
comparacion_rmse = comparar_metodos_pareado(df_val, metrica="rmse_puntual", metodo_a="doble_logistica", metodo_b="blend")

comparacion_bias

,temporada,ventana,n_ciclos_comparados,gana_doble_logistica,gana_blend,empates,tasa_victoria_blend,error_abs_medio_doble_logistica,error_abs_medio_blend,diferencia_medias,p_valor_wilcoxon,significativo_al_5pct
0,postrera,T1,48,26,22,0,0.458333,8.995214,6.731661,2.263553,0.643630,False
1,postrera,T2,50,31,19,0,0.380000,10.954505,4.651050,6.303455,0.460610,False
2,postrera,T3,43,22,20,1,0.465116,8.043492,3.231465,4.812027,0.895547,False
3,primera,T1,41,22,19,0,0.463415,12.424048,9.463797,2.960251,0.797639,False
4,primera,T2,40,18,22,0,0.550000,21.531176,8.340736,13.190441,0.157751,False
5,primera,T3,31,13,18,0,0.580645,19.159798,8.781269,10.378529,0.327150,False


In [112]:
comparacion_rmse

,temporada,ventana,n_ciclos_comparados,gana_doble_logistica,gana_blend,empates,tasa_victoria_blend,error_abs_medio_doble_logistica,error_abs_medio_blend,diferencia_medias,p_valor_wilcoxon,significativo_al_5pct
0,postrera,T1,48,27,21,0,0.437500,0.102220,0.092680,0.009541,0.955433,False
1,postrera,T2,50,31,19,0,0.380000,0.088665,0.075269,0.013396,0.187684,False
2,postrera,T3,43,23,19,1,0.441860,0.082599,0.058135,0.024464,0.639150,False
3,primera,T1,41,22,19,0,0.463415,0.109485,0.100755,0.008731,0.948901,False
4,primera,T2,40,17,23,0,0.575000,0.148015,0.087052,0.060963,0.259141,False
5,primera,T3,31,15,16,0,0.516129,0.159892,0.092817,0.067075,0.308174,False


In [108]:
def graficar_distribucion_errores(df_val: pd.DataFrame, metrica: str = "bias_integrado"):
    fig = go.Figure()
    colores = {"doble_logistica": "#d62728", "perfil_temporada": "#2ca02c", "blend": "#9467bd"}

    for temporada in df_val["temporada"].unique():
        for ventana in sorted(df_val["ventana"].unique()):
            sub = df_val[(df_val["temporada"] == temporada) & (df_val["ventana"] == ventana)]
            for metodo, color in colores.items():
                valores = sub[sub["metodo"] == metodo][metrica].abs()
                if valores.empty:
                    continue
                fig.add_trace(go.Box(
                    y=valores, name=f"{metodo}",
                    x=[f"{temporada}-{ventana}"] * len(valores),
                    marker_color=color, legendgroup=metodo,
                    showlegend=(temporada == df_val["temporada"].unique()[0] and ventana == sorted(df_val["ventana"].unique())[0]),
                ))

    fig.update_layout(
        title=f"Distribución de |{metrica}| por método, temporada y ventana",
        yaxis_title=f"|{metrica}|", boxmode="group",
        template="plotly_white", width=1200, height=550,
    )
    fig.show()

graficar_distribucion_errores(df_val, metrica="bias_integrado")

In [109]:
def encontrar_casos_donde_gana_metodo(
    df_val: pd.DataFrame, metodo_ganador: str, metodo_perdedor: str, metrica: str = "bias_integrado"
) -> pd.DataFrame:
    pivot = df_val.pivot_table(
        index=["id_ciclo", "id_parcela", "sos", "temporada", "ventana"],
        columns="metodo", values=metrica, aggfunc="first"
    ).dropna(subset=[metodo_ganador, metodo_perdedor])

    total_comparaciones = len(pivot)
    pivot["gana_" + metodo_ganador] = pivot[metodo_ganador].abs() < pivot[metodo_perdedor].abs()
    casos = pivot[pivot["gana_" + metodo_ganador]].reset_index()

    print(f"{metodo_ganador} gana en {len(casos)} de {total_comparaciones} comparaciones "
          f"({len(casos) / total_comparaciones:.1%})")

    return casos

casos_donde_gana_dl = encontrar_casos_donde_gana_metodo(df_val, "doble_logistica", "blend")

doble_logistica gana en 132 de 253 comparaciones (52.2%)


In [127]:
resumen_robusto = df_val_limpio.groupby(["temporada", "ventana", "metodo"]).agg(
    bias_abs_media=("bias_integrado", lambda x: x.abs().mean()),
    bias_abs_mediana=("bias_integrado", lambda x: x.abs().median()),
    bias_abs_p90=("bias_integrado", lambda x: x.abs().quantile(0.9)),
).reset_index()
resumen_robusto.sort_values(["temporada", "ventana", "bias_abs_mediana"])

,temporada,ventana,metodo,bias_abs_media,bias_abs_mediana,bias_abs_p90
0,postrera,T1,blend,4.089075,3.352726,8.264278
1,postrera,T1,doble_logistica,4.624138,3.443347,9.442631
2,postrera,T1,perfil_temporada,5.996601,5.131187,12.778296
4,postrera,T2,doble_logistica,1.666433,0.758294,4.774521
3,postrera,T2,blend,1.833344,1.315965,4.099809
5,postrera,T2,perfil_temporada,2.892998,2.644221,6.233104
6,postrera,T3,blend,0.822200,0.103868,3.339711
7,postrera,T3,doble_logistica,1.728776,0.119302,3.045455
8,postrera,T3,perfil_temporada,1.330557,0.579978,3.430785
10,primera,T1,doble_logistica,3.734042,2.672122,8.664121


In [114]:
def calcular_razon_cola(df_resumen: pd.DataFrame) -> pd.DataFrame:
    """
    Razón p90/mediana como indicador de riesgo de cola: valores altos
    indican que el método tiene buen desempeño típico pero es propenso
    a fallos ocasionales severos (heavy-tailed), relevante para un
    sistema de alerta temprana donde el peor caso importa tanto como
    el caso promedio.
    """
    df = df_resumen.copy()
    df["razon_cola_p90_mediana"] = df["bias_abs_p90"] / df["bias_abs_mediana"]
    return df.sort_values(["temporada", "ventana", "razon_cola_p90_mediana"])

razon_cola = calcular_razon_cola(resumen_robusto)
razon_cola

,temporada,ventana,metodo,bias_abs_media,bias_abs_mediana,bias_abs_p90,razon_cola_p90_mediana
2,postrera,T1,perfil_temporada,8.285858,6.273543,17.316641,2.760265
0,postrera,T1,blend,6.731661,4.609495,13.759881,2.985117
1,postrera,T1,doble_logistica,8.995214,4.673812,24.736249,5.292522
5,postrera,T2,perfil_temporada,5.588955,3.381660,12.111234,3.581446
3,postrera,T2,blend,4.651050,2.022994,12.204246,6.032765
4,postrera,T2,doble_logistica,10.954505,1.450998,20.703010,14.268119
8,postrera,T3,perfil_temporada,4.106815,1.913170,11.636549,6.082340
6,postrera,T3,blend,3.231465,1.169056,9.040504,7.733167
7,postrera,T3,doble_logistica,8.043492,0.696207,13.336441,19.155854
11,primera,T1,perfil_temporada,10.323638,4.737005,28.253071,5.964332


In [115]:
def identificar_ciclos_cola(
    df_val: pd.DataFrame, metodo: str, percentil: float = 0.90
) -> pd.DataFrame:
    """
    Aísla los ciclos en el percentil superior de |bias_integrado| para
    un método dado, para inspeccionar si comparten características
    (duración de ciclo corta, parcela específica, año) que expliquen
    el riesgo de cola.
    """
    sub = df_val[df_val["metodo"] == metodo].copy()
    sub["bias_abs"] = sub["bias_integrado"].abs()
    umbral = sub["bias_abs"].quantile(percentil)
    cola = sub[sub["bias_abs"] >= umbral].sort_values("bias_abs", ascending=False)
    return cola[["id_ciclo", "id_parcela", "sos", "temporada", "ventana", "bias_abs"]]

cola_doble_log = identificar_ciclos_cola(df_val, "doble_logistica")
cola_doble_log

,id_ciclo,id_parcela,sos,temporada,ventana,bias_abs
231,439,2,2021-04-20,primera,T2,296.906821
450,473,5,2022-03-29,postrera,T2,272.677596
297,446,2,2025-04-02,primera,T3,193.630388
453,473,5,2022-03-29,postrera,T3,153.797054
348,455,3,2025-04-21,primera,T2,123.141075
234,439,2,2021-04-20,primera,T3,105.068176
498,482,6,2021-06-18,primera,T2,102.312549
291,446,2,2025-04-02,primera,T1,101.582236
339,454,3,2024-05-29,primera,T2,85.144822
351,455,3,2025-04-21,primera,T3,85.100045


In [124]:
from scipy.stats import levene

def comparar_variabilidad(df_val: pd.DataFrame, temporada: str, ventana: str) -> dict:
    """
    Test de Levene sobre |bias_integrado|: evalúa si la dispersión del
    error difiere significativamente entre métodos, complementando el
    test de Wilcoxon (que compara medianas) con evidencia sobre
    consistencia/riesgo de cola.
    """
    sub = df_val[(df_val["temporada"] == temporada) & (df_val["ventana"] == ventana)]
    grupos = {m: sub[sub["metodo"] == m]["bias_integrado"].abs().dropna() for m in sub["metodo"].unique()}

    stat, p_valor = levene(*grupos.values())
    return {
        "temporada": temporada, "ventana": ventana,
        "statistic": stat, "p_valor": p_valor,
        "varianzas_distintas_al_5pct": p_valor < 0.05,
        **{f"std_{m}": v.std() for m, v in grupos.items()},
    }

resultados_levene = pd.DataFrame([
    comparar_variabilidad(df_val, t, v)
    for t in df_val["temporada"].unique()
    for v in df_val["ventana"].unique()
])
resultados_levene

,temporada,ventana,statistic,p_valor,varianzas_distintas_al_5pct,std_doble_logistica,std_perfil_temporada,std_blend
0,postrera,T1,2.413589,0.093183,False,11.005787,6.773405,6.209059
1,postrera,T2,1.385195,0.253522,False,39.001624,6.034191,5.938238
2,postrera,T3,1.356307,0.261345,False,25.358917,5.737216,5.403750
3,primera,T1,0.646240,0.525826,False,20.322032,15.084953,14.694352
4,primera,T2,2.390886,0.096008,False,52.640785,14.603110,15.022505
5,primera,T3,1.569903,0.213709,False,40.753998,15.115245,14.313302


In [123]:
from scipy.stats import levene

def comparar_variabilidad_robusta(df_val: pd.DataFrame, temporada: str, ventana: str) -> dict:
    """
    Test de Brown-Forsythe (Levene centrado en mediana, center='median')
    en vez de Levene clásico (centrado en media). Más robusto cuando las
    distribuciones tienen colas pesadas / outliers extremos, como es el
    caso aquí — el Levene clásico pierde potencia porque la media de
    cada grupo ya está distorsionada por los mismos outliers que se
    intenta detectar.
    """
    sub = df_val[(df_val["temporada"] == temporada) & (df_val["ventana"] == ventana)]
    grupos = {m: sub[sub["metodo"] == m]["bias_integrado"].abs().dropna() for m in sub["metodo"].unique()}

    stat, p_valor = levene(*grupos.values(), center="median")
    return {
        "temporada": temporada, "ventana": ventana,
        "statistic_bf": stat, "p_valor_bf": p_valor,
        "varianzas_distintas_al_5pct_bf": p_valor < 0.05,
    }

resultados_bf = pd.DataFrame([
    comparar_variabilidad_robusta(df_val, t, v)
    for t in df_val["temporada"].unique()
    for v in df_val["ventana"].unique()
])
resultados_bf

,temporada,ventana,statistic_bf,p_valor_bf,varianzas_distintas_al_5pct_bf
0,postrera,T1,2.413589,0.093183,False
1,postrera,T2,1.385195,0.253522,False
2,postrera,T3,1.356307,0.261345,False
3,primera,T1,0.646240,0.525826,False
4,primera,T2,2.390886,0.096008,False
5,primera,T3,1.569903,0.213709,False


In [118]:
def analizar_reincidencia_por_parcela(cola_doble_log: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa los ciclos en la cola de peor error por id_parcela, para
    detectar si el riesgo de cola se concentra en unas pocas parcelas
    problemáticas (posible causa estructural: calidad de datos, mezcla
    de cultivo, SOS mal detectado) en vez de estar disperso aleatoriamente
    entre el catálogo completo.
    """
    resumen = cola_doble_log.groupby("id_parcela").agg(
        n_apariciones_en_cola=("id_ciclo", "count"),
        n_ciclos_distintos=("id_ciclo", "nunique"),
        bias_abs_medio_en_cola=("bias_abs", "mean"),
        bias_abs_maximo=("bias_abs", "max"),
    ).sort_values("n_apariciones_en_cola", ascending=False)
    return resumen

reincidencia = analizar_reincidencia_por_parcela(cola_doble_log)
reincidencia

,n_apariciones_en_cola,n_ciclos_distintos,bias_abs_medio_en_cola,bias_abs_maximo
id_parcela,,,,
5,8,4,79.842414,272.677596
2,7,3,117.677003,296.906821
3,4,3,89.978124,123.141075
4,3,2,47.977857,53.036151
6,2,2,72.638200,102.312549
7,2,1,40.834016,48.701188


In [119]:
def inspeccionar_ciclo_problematico(id_ciclo: int, indice_nombre: str = "EVI"):
    """
    Trae el detalle completo de un ciclo específico desde BD para
    inspección manual: duración real, r2 del ajuste doble-logístico,
    y si hay señales de ciclo siguiente muy próximo.
    """
    ciclos_df = obtener_ciclos_desde_bd()
    fila = ciclos_df[ciclos_df["id_ciclo"] == id_ciclo]
    if fila.empty:
        print(f"Ciclo {id_ciclo} no encontrado.")
        return
    fila = fila.iloc[0]
    duracion = (fila["eos"] - fila["sos"]).days
    print(f"Parcela {fila['id_parcela']} | temporada {fila['temporada']} | "
          f"SOS {fila['sos'].date()} → EOS {fila['eos'].date()} | duración {duracion} días")

    # Ciclo siguiente de la misma parcela, para chequear proximidad
    otros = ciclos_df[(ciclos_df["id_parcela"] == fila["id_parcela"]) & (ciclos_df["sos"] > fila["eos"])]
    if not otros.empty:
        siguiente = otros.sort_values("sos").iloc[0]
        gap = (siguiente["sos"] - fila["eos"]).days
        print(f"Siguiente ciclo: SOS {siguiente['sos'].date()} (temporada {siguiente['temporada']}) | gap: {gap} días")

# Ejemplo con los peores casos de tu tabla
for id_ciclo in [439, 473, 446]:
    inspeccionar_ciclo_problematico(id_ciclo)
    print()

Parcela 2 | temporada primera | SOS 2021-04-20 → EOS 2022-03-24 | duración 338 días
Siguiente ciclo: SOS 2022-04-05 (temporada primera) | gap: 12 días

Parcela 5 | temporada postrera | SOS 2022-03-29 → EOS 2023-05-14 | duración 411 días
Siguiente ciclo: SOS 2023-06-06 (temporada primera) | gap: 23 días

Parcela 2 | temporada primera | SOS 2025-04-02 → EOS 2026-01-27 | duración 300 días
Siguiente ciclo: SOS 2026-02-09 (temporada postrera) | gap: 13 días



In [120]:
def diagnosticar_duraciones_anomalas(
    duracion_maxima_esperada: int = 150,  # margen sobre SOS+120
) -> pd.DataFrame:
    """
    Identifica ciclos persistidos en produccion_acumulada_ciclo cuya
    duración SOS→EOS excede lo agronómicamente esperado (~120 días +
    margen), señal de que segmentar_ciclos no encontró un valle válido
    dentro de la ventana fenológica normal — ya sea por ausencia de
    valle suficientemente profundo (prominencia_min insuficiente) o por
    gaps de datos que distorsionan la serie suavizada.
    """
    ciclos_df = obtener_ciclos_desde_bd()
    ciclos_df["duracion_dias"] = (ciclos_df["eos"] - ciclos_df["sos"]).dt.days

    anomalos = ciclos_df[ciclos_df["duracion_dias"] > duracion_maxima_esperada].copy()
    anomalos = anomalos.sort_values("duracion_dias", ascending=False)

    print(f"Ciclos anómalos (duración > {duracion_maxima_esperada} días): "
          f"{len(anomalos)} de {len(ciclos_df)} ({len(anomalos)/len(ciclos_df):.1%})")
    print(f"\nPor parcela:")
    print(anomalos.groupby("id_parcela")["id_ciclo"].count().sort_values(ascending=False))

    return anomalos[["id_ciclo", "id_parcela", "temporada", "sos", "eos", "duracion_dias"]]

anomalos = diagnosticar_duraciones_anomalas()
anomalos

Ciclos anómalos (duración > 150 días): 34 de 112 (30.4%)

Por parcela:
id_parcela
5    6
3    5
7    4
6    4
2    4
0    3
1    3
4    3
8    2
Name: id_ciclo, dtype: int64


,id_ciclo,id_parcela,temporada,sos,eos,duracion_dias
68,473,5,postrera,2022-03-29,2023-05-14,411
45,450,3,primera,2021-04-06,2022-04-25,384
34,439,2,primera,2021-04-20,2022-03-24,338
54,459,4,primera,2021-06-03,2022-04-12,313
69,474,5,primera,2023-06-06,2024-04-14,313
71,476,5,primera,2025-05-10,2026-03-18,312
53,458,4,primera,2020-07-01,2021-05-08,311
49,454,3,primera,2024-05-29,2025-03-27,302
41,446,2,primera,2025-04-02,2026-01-27,300
70,475,5,primera,2024-06-22,2025-04-09,291


In [131]:
def graficar_serie_con_segmentacion_cruda(parcela: str, indice_nombre: str = "EVI"):
    """
    Grafica la serie suavizada completa con los valles detectados por
    segmentar_ciclos marcados, para inspeccionar visualmente por qué un
    segmento resultó anómalamente largo (ausencia de valle, gap de
    datos, prominencia insuficiente).
    """
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=serie.index, y=serie.values, mode="lines",
                              line=dict(color="#1f77b4", width=1.5), name="Serie suavizada"))

    for inicio, fin in segmentos:
        for f in (inicio, fin):
            fig.add_vline(x=f, line_width=1, line_dash="dot", line_color="red")

    fig.update_layout(title=f"Segmentación cruda: {indice_nombre} en {parcela}",
                       template="plotly_white", width=1300, height=450)
    fig.show()

graficar_serie_con_segmentacion_cruda("id_5", "EVI")

In [122]:
def filtrar_ciclos_validos(df_val: pd.DataFrame, duracion_maxima_esperada: int = 150) -> pd.DataFrame:
    """
    Excluye de la validación los ciclos cuya duración SOS-EOS excede lo
    agronómicamente plausible, evitando que fallas de segmentación (no
    de método de extrapolación) contaminen la comparación de métodos.
    """
    ciclos_df = obtener_ciclos_desde_bd()
    ciclos_df["duracion_dias"] = (ciclos_df["eos"] - ciclos_df["sos"]).dt.days
    ids_validos = ciclos_df[ciclos_df["duracion_dias"] <= duracion_maxima_esperada]["id_ciclo"]

    antes = df_val["id_ciclo"].nunique()
    df_filtrado = df_val[df_val["id_ciclo"].isin(ids_validos)]
    despues = df_filtrado["id_ciclo"].nunique()

    print(f"Ciclos: {antes} → {despues} ({antes - despues} excluidos por duración anómala)")
    return df_filtrado

df_val_limpio = filtrar_ciclos_validos(df_val)

Ciclos: 92 → 63 (29 excluidos por duración anómala)


In [128]:
comparacion_bias_limpia = comparar_metodos_pareado(
    df_val_limpio, metrica="bias_integrado", metodo_a="doble_logistica", metodo_b="blend"
)
comparacion_bias_limpia

,temporada,ventana,n_ciclos_comparados,gana_doble_logistica,gana_blend,empates,tasa_victoria_blend,error_abs_medio_doble_logistica,error_abs_medio_blend,diferencia_medias,p_valor_wilcoxon,significativo_al_5pct
0,postrera,T1,32,18,14,0,0.437500,4.624138,4.089075,0.535063,0.832040,False
1,postrera,T2,34,23,11,0,0.323529,1.666433,1.833344,-0.166911,0.069589,False
2,postrera,T3,28,13,14,1,0.500000,1.728776,0.822200,0.906576,0.904384,False
3,primera,T1,28,17,11,0,0.392857,3.734042,4.187320,-0.453278,0.350163,False
4,primera,T2,27,15,12,0,0.444444,2.668680,2.391331,0.277349,0.767608,False
5,primera,T3,18,8,10,0,0.555556,2.296100,1.959275,0.336825,0.966118,False


In [129]:
def diagnosticar_causa_segmentacion_fallida(id_ciclo: int) -> None:
    """
    Para un ciclo con duración anómala, verifica si la causa es
    cobertura de datos insuficiente (gap prolongado sin observaciones
    Sentinel-2 reales) en el tramo donde debería estar el valle, lo cual
    aplana artificialmente la serie Whittaker y reduce la prominencia
    detectable por find_peaks.
    """
    ciclos_df = obtener_ciclos_desde_bd()
    fila = ciclos_df[ciclos_df["id_ciclo"] == id_ciclo].iloc[0]
    col_parcela = f"id_{fila['id_parcela']}"

    ventana_esperada_eos = fila["sos"] + pd.Timedelta(days=120)
    tramo_crudo = dfs_raw["EVI"].loc[
        fila["sos"] + pd.Timedelta(days=80): ventana_esperada_eos + pd.Timedelta(days=30), col_parcela
    ]
    n_obs_reales = tramo_crudo.notna().sum()
    n_dias_totales = len(tramo_crudo)

    print(f"Ciclo {id_ciclo} (parcela {fila['id_parcela']}): "
          f"{n_obs_reales} observaciones Sentinel-2 reales de {n_dias_totales} días "
          f"({n_obs_reales/n_dias_totales:.1%}) en la ventana donde se esperaba el valle (día 80-150 post-SOS)")

for id_ciclo in [473, 450, 439, 459]:
    diagnosticar_causa_segmentacion_fallida(id_ciclo)

Ciclo 473 (parcela 5): 14 observaciones Sentinel-2 reales de 71 días (19.7%) en la ventana donde se esperaba el valle (día 80-150 post-SOS)
Ciclo 450 (parcela 3): 15 observaciones Sentinel-2 reales de 71 días (21.1%) en la ventana donde se esperaba el valle (día 80-150 post-SOS)
Ciclo 439 (parcela 2): 14 observaciones Sentinel-2 reales de 71 días (19.7%) en la ventana donde se esperaba el valle (día 80-150 post-SOS)
Ciclo 459 (parcela 4): 14 observaciones Sentinel-2 reales de 71 días (19.7%) en la ventana donde se esperaba el valle (día 80-150 post-SOS)


In [130]:
def marcar_ciclos_duracion_anomala(duracion_maxima_esperada: int = 150) -> pd.DataFrame:
    """
    Identifica ciclos en produccion_acumulada_ciclo con duración SOS-EOS
    fuera de rango agronómico plausible, para excluirlos de generación
    de predicciones o marcarlos con baja confianza en el observatorio.
    No modifica la BD; retorna la lista para revisión antes de decidir
    la acción (exclusión, recalibración de segmentación, o bandera de
    baja confianza en el dashboard).
    """
    ciclos_df = obtener_ciclos_desde_bd()
    ciclos_df["duracion_dias"] = (ciclos_df["eos"] - ciclos_df["sos"]).dt.days
    return ciclos_df[ciclos_df["duracion_dias"] > duracion_maxima_esperada]